# TryFit - AI Virtual Try-On Pipeline (0 TL Colab T4 GPU)
Bu notebook, Trendyol ürün fotoğraflarını dekupe edip CatVTON / IDM-VTON ile fotogerçekçi giydirme çıktıları üretir.

### Adımlar:
1. **GPU & Ortam Kurulumu**: PyTorch, Gradio Client, Diffusers, ONNX
2. **Trendyol Arka Plan Kaldırma (Background Removal)**: Ürün fotoğraflarını temizleme
3. **CatVTON / IDM-VTON Inference**: 10 Küratörlü Trendyol kıyafetini model üzerinde deneme
4. **Çok Açılı (2.5D) Çıktı Kaydı**: Ön, 45°, Yan ve Arka görsellerini TryFit backend'e aktarma

In [ ]:
# 1. GPU Kontrolü & Gereksinimlerin Yüklenmesi
!nvidia-smi
!pip install -q diffusers transformers accelerate gradio_client onnxruntime torchvision

In [ ]:
# 2. Trendyol Background Removal (Dekupe Kütüphanesi)
from PIL import Image
import torch
import urllib.request

print("PyTorch CUDA Kullanılabilir:", torch.cuda.is_available())
print("Cihaz Adı:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# 3. CatVTON / IDM-VTON Test Inference Fonksiyonu
from gradio_client import Client, handle_file

def run_vton_tryon(model_image_path, garment_image_path, garment_description="clothing"):
    client = Client("yisol-IDM-VTON")
    result = client.predict(
        dict={"background": handle_file(model_image_path), "layers": [], "composite": None},
        garm_img=handle_file(garment_image_path),
        garment_des=garment_description,
        is_checked=True,
        is_checked_crop=False,
        denoise_steps=25,
        seed=42,
        api_name="/tryon"
    )
    return result[0] # Try-on üretilen görselin geçici dosya yolu

print("Inference fonksiyonu hazır!")

In [ ]:
# 4. 10 Küratörlü Trendyol Ürünü Toplu Üretim Scripti
SAMPLE_DATASET = [
    {"id": "ty-hoodie-01", "desc": "Oversize Black Hoodie", "cat": "Top"},
    {"id": "ty-hoodie-02", "desc": "Melange Grey Zip Hoodie", "cat": "Top"},
    {"id": "ty-hoodie-03", "desc": "Ecru Knit Sweatshirt", "cat": "Top"},
    {"id": "ty-dress-01", "desc": "Red Satin Cocktail Dress", "cat": "Dress"},
    {"id": "ty-dress-02", "desc": "Floral Chiffon Mini Dress", "cat": "Dress"},
    {"id": "ty-dress-03", "desc": "Black Ribbed Midi Dress", "cat": "Dress"},
    {"id": "ty-coat-01", "desc": "Beige Classic Trench Coat", "cat": "Outerwear"},
    {"id": "ty-coat-02", "desc": "Black Biker Leather Jacket", "cat": "Outerwear"},
    {"id": "ty-pants-01", "desc": "Light Blue Mom Jeans", "cat": "Bottom"},
    {"id": "ty-pants-02", "desc": "Wide Leg Black Palazzo Pants", "cat": "Bottom"}
]

print(f"{len(SAMPLE_DATASET)} adet küratörlü ürün için pipeline hazırlandı.")